# Entrenar la voz propia de Asia (Piper) en Colab

Para Colab gratuito con GPU T4: **Entorno de ejecución → Cambiar tipo → T4 GPU**, y luego ejecuta las celdas **en orden**.
Fine-tuning de Piper (piper1-gpl) desde un checkpoint medium en español, con los checkpoints en tu Drive para poder
**reanudar** en otra sesión (Colab corta hacia las 4 h): vuelve a ejecutar todo y sigue desde el último.

Antes de empezar:
1. Sube a Drive, en `MyDrive/asia_voz/entrada/`, el zip del dataset (`fa_normal_22050.zip` o `fa_atenuado_22050.zip`, de `dataset_final.py`).
2. La voz es de otra persona, con permiso y para uso **privado**. El zip solo debe subirse a tu Drive con su visto bueno.
3. La última celda borra los audios de Drive y de Colab. No ejecutes el cuaderno compartido ni lo publiques con salidas.
4. **Primera vez: prueba de humo.** Pon `SMOKE = True` en la configuración y ejecuta todo (unos 10–15 min más las instalaciones).
   Usa 20 frases y comprueba con ✅/❌ que (a) se respeta `max_time`, (b) se reanuda con `--ckpt_path` desde `last.ckpt` y
   (c) se exporta el `.onnx` y se sintetiza una frase. Si todo sale ✅, pon `SMOKE = False` y reinicia el entorno (Entorno de ejecución → Reiniciar).
   El modo humo usa rutas propias (`checkpoints_humo`, `/content/humo`) y no toca los checkpoints ni los audios reales.

Pasos y tiempos estimados: `docs/VOZ_PROPIA.md` en el repositorio de Asia.

## 0. Configuración

In [ ]:
# Lo que se cambia de una sesión a otra.
SMOKE = False                         # True: prueba de humo (20 frases, ver la sección 4b); False: entrenamiento real
DATASET_ZIP = "fa_normal_22050.zip"   # o "fa_atenuado_22050.zip" (silencios atenuados)
VOICE_NAME = "es_MX-propia"
BASE = "ald"                          # "ald" (es_MX, hombre; recomendado) o "sharvard" (plan B: es_ES, 2 hablantes)
ESPEAK_VOICE = "es-419"               # español de América Latina
BATCH_SIZE = 16
MAX_EPOCHS = 1000                     # TOTAL, contando las de sesiones anteriores (un checkpoint reanudado cuenta las suyas)
SESSION_MAX_TIME = "00:03:30:00"      # DD:HH:MM:SS; para antes del corte de Colab (~4 h) y deja un último checkpoint
SYNC_EVERY_S = 300                    # cada cuánto se copia last.ckpt a Drive

# Prueba de humo
SMOKE_PHRASES = 20
SMOKE_MAX_TIME = "00:00:03:00"        # 3 min
SMOKE_STARTUP_MARGIN_S = 240          # lo que puede tardar de más el arranque y el último checkpoint

DRIVE_DIR = "/content/drive/MyDrive/asia_voz"
LOCAL = "/content"

# Checkpoints de base (rhasspy/piper-checkpoints, es). Se bajan a Colab, no a tu PC.
HF = "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/es"
BASES = {
    "ald": {
        "path": "es_MX/ald/medium", "ckpt": "epoch%3D9999-step%3D1753600.ckpt",
        "sha256": "77ee30f9a382594123ec4fb3ca53b437612819a44b3e3dc490f39f5a2dfefcce",
    },
    "sharvard": {
        "path": "es_ES/sharvard/medium", "ckpt": "epoch%3D4899-step%3D215600.ckpt",
        "sha256": "cae76056c432f1a477705af1ebec55b73b586865e73b256d4bdc6fdaea8296c9",
    },
}
assert BASE in BASES, BASE
print("MODO HUMO: 20 frases, sin tocar los checkpoints reales" if SMOKE else "Entrenamiento real")

## 1. GPU y Drive

In [ ]:
import subprocess, sys
print(sys.version)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

for folder in ("entrada", "base", "checkpoints", "salida"):
    os.makedirs(f"{DRIVE_DIR}/{folder}", exist_ok=True)

print(os.listdir(f"{DRIVE_DIR}/entrada"))

## 2. Instalación (solo en Colab)

In [ ]:
!apt-get -qq install -y build-essential cmake ninja-build espeak-ng aria2 > /dev/null
%cd /content
!git clone -q https://github.com/OHF-voice/piper1-gpl.git
%cd /content/piper1-gpl
!python3 -m pip install -q -e '.[train]'
!./build_monotonic_align.sh
!python3 setup.py build_ext --inplace

In [ ]:
# Con PyTorch 2.6 o posterior, cargar un checkpoint falla por torch.load(weights_only=True)
# (discusión #138 de piper1-gpl). Esta variable de PyTorch lo desactiva sin tocar el código de Piper.
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

import torch
print("torch", torch.__version__, "· GPU:", torch.cuda.is_available())

## 3. Dataset y checkpoint de base

In [ ]:
import glob, shutil, zipfile

zip_path = f"{DRIVE_DIR}/entrada/{DATASET_ZIP}"
assert os.path.exists(zip_path), f"Falta {zip_path}: súbelo a Drive (carpeta asia_voz/entrada)"

shutil.rmtree(f"{LOCAL}/dataset", ignore_errors=True)

with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(f"{LOCAL}/dataset")

lines = open(f"{LOCAL}/dataset/metadata.csv", encoding="utf-8").read().splitlines()
wavs = glob.glob(f"{LOCAL}/dataset/wavs/*.wav")
print(len(lines), "frases;", len(wavs), "wavs")
assert len(lines) == len(wavs) > 0

DATA = f"{LOCAL}/dataset"

if SMOKE:
    # Solo las primeras SMOKE_PHRASES frases, en otra carpeta.
    DATA = f"{LOCAL}/dataset_humo"
    shutil.rmtree(DATA, ignore_errors=True)
    os.makedirs(f"{DATA}/wavs")
    lines = lines[:SMOKE_PHRASES]

    for line in lines:
        shutil.copy(f"{LOCAL}/dataset/wavs/{line.split('|')[0]}", f"{DATA}/wavs/")

    open(f"{DATA}/metadata.csv", "w", encoding="utf-8").write("".join(line + chr(10) for line in lines))
    print("Modo humo:", len(lines), "frases en", DATA)

In [ ]:
import hashlib

info = BASES[BASE]
base_dir = f"{DRIVE_DIR}/base/{BASE}"
os.makedirs(base_dir, exist_ok=True)
base_ckpt = f"{base_dir}/base.ckpt"

# aria2c y no wget: wget corrompe las descargas de Hugging Face en Colab (el archivo empieza por "<" en vez de "PK").
if not os.path.exists(base_ckpt):
    !aria2c -x 8 -s 8 -q -d {base_dir} -o base.ckpt "{HF}/{info['path']}/{info['ckpt']}"

with open(base_ckpt, "rb") as f:
    assert f.read(2) == b"PK", "La descarga no es un checkpoint (¿página de error?)"

digest = hashlib.sha256()

with open(base_ckpt, "rb") as f:
    for block in iter(lambda: f.read(1 << 24), b""):
        digest.update(block)

assert digest.hexdigest() == info["sha256"], "sha256 distinto: descarga corrupta, bórrala y repite"
print("base:", BASE, f"{os.path.getsize(base_ckpt) / 1e6:.0f} MB, sha256 correcto")

## 4. Entrenamiento (reanudable)

- Primera vez: `--model.warmstart_ckpt` copia solo los pesos de la base y empieza en la época 0 (los checkpoints viejos de
  `piper-checkpoints` no se pueden reanudar con `--ckpt_path`, solo usar de base).
- Si ya hay un `last.ckpt` en Drive: `--ckpt_path` reanuda la época, el optimizador y el scheduler. `MAX_EPOCHS` debe incluir las ya hechas.
- Piper reescribe `last.ckpt` en cada época. Entrena en disco local y copia `last.ckpt` a Drive cada `SYNC_EVERY_S` s
  (copia a un archivo temporal y renombra, para no dejar un checkpoint a medias si Colab se corta).
- Escucha a las 300, 500 y 1000 épocas (sección 5): con tan poco audio suele bastar antes de 2000.
- Plan B (`BASE = "sharvard"`, dos hablantes): cargar sus pesos en un modelo de un hablante puede fallar por la forma de las capas;
  no está probado. Si falla, revisa `docs/VOZ_PROPIA.md`.

In [ ]:
import subprocess, threading, time, glob

if SMOKE:
    # Rutas propias: el modo humo no toca los checkpoints ni la voz reales.
    LOGS = f"{LOCAL}/humo"
    CKPT_DRIVE = f"{DRIVE_DIR}/checkpoints_humo/last.ckpt"
    CONFIG = f"{LOCAL}/humo/{VOICE_NAME}-medium.onnx.json"
    ONNX = f"{LOCAL}/humo/{VOICE_NAME}-medium.onnx"
    shutil.rmtree(LOGS, ignore_errors=True)
    shutil.rmtree(os.path.dirname(CKPT_DRIVE), ignore_errors=True)
    os.makedirs(LOGS)
else:
    LOGS = f"{LOCAL}/entrenamiento"
    CKPT_DRIVE = f"{DRIVE_DIR}/checkpoints/last.ckpt"
    CONFIG = f"{DRIVE_DIR}/salida/{VOICE_NAME}-medium.onnx.json"
    ONNX = f"{DRIVE_DIR}/salida/{VOICE_NAME}-medium.onnx"

os.makedirs(os.path.dirname(CKPT_DRIVE), exist_ok=True)


def newest_last():
    found = glob.glob(f"{LOGS}/lightning_logs/version_*/checkpoints/last.ckpt")
    return max(found, key=os.path.getmtime) if found else None


def sync():
    """last.ckpt local -> Drive, de forma atómica."""
    source = newest_last()

    if source:
        shutil.copyfile(source, CKPT_DRIVE + ".tmp")
        os.replace(CKPT_DRIVE + ".tmp", CKPT_DRIVE)
        print(f"[sync] {os.path.basename(os.path.dirname(os.path.dirname(source)))} -> Drive", time.strftime("%H:%M:%S"))


def run_training(max_epochs, max_time, resume, timeout_s=None):
    """Entrena; devuelve (código de salida, segundos, True si hubo que matarlo por timeout_s)."""
    start = ["--ckpt_path", CKPT_DRIVE] if resume else ["--model.warmstart_ckpt", base_ckpt]
    command = [
        sys.executable, "-m", "piper.train", "fit",
        "--data.voice_name", VOICE_NAME,
        "--data.csv_path", f"{DATA}/metadata.csv",
        "--data.audio_dir", f"{DATA}/wavs/",
        "--model.sample_rate", "22050",
        "--data.espeak_voice", ESPEAK_VOICE,
        "--data.cache_dir", f"{LOCAL}/cache/",
        "--data.config_path", CONFIG,
        "--data.batch_size", str(BATCH_SIZE),
        "--trainer.max_epochs", str(max_epochs),
        "--trainer.max_time", max_time,
        "--trainer.default_root_dir", LOGS,
    ] + start
    print("Reanudando desde" if resume else "Empezando desde", CKPT_DRIVE if resume else base_ckpt)
    print(" ".join(command))

    began = time.time()
    process = subprocess.Popen(command, cwd="/content/piper1-gpl", stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    killed = []
    watchdog = None

    if timeout_s:
        watchdog = threading.Timer(timeout_s, lambda: (killed.append(True), process.kill()))
        watchdog.start()

    last_sync = time.time()

    try:
        for line in process.stdout:
            print(line, end="")

            if time.time() - last_sync > SYNC_EVERY_S:
                sync()
                last_sync = time.time()
    finally:
        process.wait()

        if watchdog:
            watchdog.cancel()

        sync()

    seconds = time.time() - began
    print("Código de salida:", process.returncode, f"· {seconds:.0f} s")
    return process.returncode, seconds, bool(killed)

In [ ]:
# Entrenamiento real (en modo humo se salta: la sección 4b hace la prueba).
if SMOKE:
    print("Modo humo: salto el entrenamiento real; sigue con la sección 4b.")
else:
    run_training(MAX_EPOCHS, SESSION_MAX_TIME, os.path.exists(CKPT_DRIVE))

## 4b. Prueba de humo (solo con `SMOKE = True`)

Con 20 frases comprueba, con ✅/❌:
- **(a)** que se respeta `max_time`: entrena con muchas épocas y un límite de 3 min, y debe parar solo;
- **(b)** que se reanuda con `--ckpt_path` desde `last.ckpt` (la copia de Drive): una época más y el contador sigue subiendo, no empieza de cero;
- **(c)** que se exporta el `.onnx` y se sintetiza una frase de prueba que suena (no está vacía).

Si algo sale ❌, no empieces el entrenamiento real.

In [ ]:
import array
import wave


def seconds_of(max_time):
    """"DD:HH:MM:SS" -> segundos."""
    days, hours, minutes, seconds = (int(part) for part in max_time.split(":"))
    return ((days * 24 + hours) * 60 + minutes) * 60 + seconds


def judge_max_time(returncode, seconds, killed, limit_s, margin_s):
    """(a) Terminó solo, sin error, ni antes de tiempo ni mucho después del límite."""
    if killed:
        return False, f"max_time no se respetó: hubo que matar el entrenamiento a los {seconds:.0f} s (límite {limit_s} s)"

    if returncode != 0:
        return False, f"el entrenamiento falló (código {returncode}) tras {seconds:.0f} s"

    if seconds < limit_s * 0.6:
        return False, f"paró a los {seconds:.0f} s, mucho antes del límite de {limit_s} s: no fue por max_time"

    if seconds > limit_s + margin_s:
        return False, f"tardó {seconds:.0f} s con un límite de {limit_s} s (+{margin_s} s de arranque)"

    return True, f"max_time respetado: paró solo a los {seconds:.0f} s (límite {limit_s} s)"


def judge_resume(returncode, before, after):
    """(b) Tras reanudar, el contador de épocas y de pasos subió (no empezó de cero)."""
    if before is None:
        return False, "no había last.ckpt en Drive para reanudar"

    if returncode != 0:
        return False, f"reanudar falló (código {returncode})"

    if after is None or after["global_step"] <= before["global_step"] or after["epoch"] <= before["epoch"]:
        return False, f"no continuó: antes {before}, después {after}"

    return True, f"reanudó desde last.ckpt: época {before['epoch']} -> {after['epoch']}, paso {before['global_step']} -> {after['global_step']}"


def judge_synth(exported, synthesized, wav_path):
    """(c) Hay .onnx y la frase sintetizada dura al menos 0,5 s y no es silencio."""
    if not exported:
        return False, "no se exportó el .onnx"

    if not synthesized or not os.path.exists(wav_path):
        return False, "no se sintetizó la frase de prueba"

    with wave.open(wav_path, "rb") as clip:
        seconds = clip.getnframes() / clip.getframerate()
        samples = array.array("h", clip.readframes(clip.getnframes()))

    peak = max((abs(sample) for sample in samples), default=0) / 32768

    if seconds < 0.5 or peak < 0.01:
        return False, f"la frase de prueba es demasiado corta o está en silencio ({seconds:.1f} s, pico {peak:.3f})"

    return True, f"onnx exportado y frase sintetizada: {seconds:.1f} s, pico {peak:.2f}"


def show(verdict):
    ok, detail = verdict
    print("✅" if ok else "❌", detail)
    return ok

In [ ]:
if not SMOKE:
    print("Modo real: no hace falta la prueba de humo.")
else:
    def checkpoint_state(path):
        if not os.path.exists(path):
            return None

        data = torch.load(path, map_location="cpu", weights_only=False)
        return {"epoch": int(data["epoch"]), "global_step": int(data["global_step"])}

    limit = seconds_of(SMOKE_MAX_TIME)
    results = []

    # (a) Muchas épocas, 3 min de límite: debe parar solo.
    code_a, seconds_a, killed_a = run_training(100000, SMOKE_MAX_TIME, False, timeout_s=limit + SMOKE_STARTUP_MARGIN_S)
    results.append(show(judge_max_time(code_a, seconds_a, killed_a, limit, SMOKE_STARTUP_MARGIN_S)))

    # (b) Reanudar desde el last.ckpt de Drive y entrenar una época más.
    before = checkpoint_state(CKPT_DRIVE)
    code_b = None

    if before:
        code_b, _, _ = run_training(before["epoch"] + 2, "00:00:10:00", True, timeout_s=600)

    results.append(show(judge_resume(code_b, before, checkpoint_state(CKPT_DRIVE))))

    # (c) Exportar y sintetizar una frase.
    wav_path = f"{LOGS}/prueba.wav"
    exported = subprocess.run([sys.executable, "-m", "piper.train.export_onnx", "--checkpoint", CKPT_DRIVE,
                               "--output-file", ONNX]).returncode == 0 and os.path.exists(ONNX)
    synthesized = exported and subprocess.run([sys.executable, "-m", "piper", "-m", ONNX, "-c", CONFIG, "-f", wav_path,
                                               "--", "Hola, soy Asia. Esta es una prueba."]).returncode == 0
    results.append(show(judge_synth(exported, synthesized, wav_path)))

    print()
    print("✅ Todo bien: pon SMOKE = False, reinicia el entorno y entrena de verdad." if all(results)
          else "❌ Hay fallos: no empieces el entrenamiento real.")

## 5. Probar y exportar a .onnx

In [ ]:
# Exporta el último checkpoint (o pon aquí otro: CKPT = "/content/drive/MyDrive/asia_voz/checkpoints/otro.ckpt").
if SMOKE:
    print("Modo humo: la exportación ya se probó en la sección 4b.")
else:
    CKPT = CKPT_DRIVE
    assert os.path.exists(CKPT), "No hay checkpoint: entrena primero"
    assert os.path.exists(CONFIG), "Falta el JSON de la voz (lo escribe el entrenamiento)"

    !python3 -m piper.train.export_onnx --checkpoint {CKPT} --output-file {ONNX}
    print(os.listdir(f"{DRIVE_DIR}/salida"))

In [ ]:
# Una frase de prueba con la voz exportada.
if not SMOKE:
    from IPython.display import Audio, display

    TEXT = "Hola, soy Asia. Esta es mi voz nueva; dime si se entiende bien."
    !python3 -m piper -m {ONNX} -c {CONFIG} -f /content/prueba.wav -- "{TEXT}"
    display(Audio("/content/prueba.wav"))

In [ ]:
# Baja los dos archivos a tu PC: van a data/voices/ de Asia (docs/VOZ_PROPIA.md, "Probarla en Asia").
if not SMOKE:
    from google.colab import files

    files.download(ONNX)
    files.download(CONFIG)

## 6. Borrar los audios

Borra el zip de `entrada/` en Drive y los audios de Colab. **No** borra los checkpoints ni la voz exportada
(`checkpoints/` y `salida/`): sin los audios no se puede seguir entrenando desde cero, pero sí reanudar con los
checkpoints si vuelves a subir el zip.

Drive manda lo borrado a la **Papelera**: vacíala en https://drive.google.com/drive/trash para que de verdad se vaya.

In [ ]:
DELETE_AUDIO = True   # pon False si todavía vas a reanudar el entrenamiento con este zip

if SMOKE:
    # El modo humo borra solo lo suyo: lo demás (el zip) se necesita para el entrenamiento real.
    shutil.rmtree(f"{LOCAL}/dataset_humo", ignore_errors=True)
    shutil.rmtree(f"{DRIVE_DIR}/checkpoints_humo", ignore_errors=True)
    print("Modo humo: borré dataset_humo y checkpoints_humo; el zip de entrada sigue en Drive.")
elif DELETE_AUDIO:
    shutil.rmtree(f"{DRIVE_DIR}/entrada", ignore_errors=True)
    shutil.rmtree(f"{LOCAL}/dataset", ignore_errors=True)
    shutil.rmtree(f"{LOCAL}/cache", ignore_errors=True)
    os.makedirs(f"{DRIVE_DIR}/entrada", exist_ok=True)
    left = [p for p in glob.glob(f"{DRIVE_DIR}/**/*.wav", recursive=True) + glob.glob(f"{DRIVE_DIR}/**/*.zip", recursive=True)]
    print("Audios que quedan en Drive:", left)
    print("Ahora vacía la Papelera de Drive: https://drive.google.com/drive/trash")
else:
    print("No se borró nada (DELETE_AUDIO = False).")